# Praktikum 4 : perbandingan Annoy, FAISS, dan HNSW

In [1]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib
import pandas as pd
from IPython.display import display

# 1. Buat dataset 1 juta data 5D
np.random.seed(42)
import pandas as pd
df = pd.read_csv('../../dataset_pbl/cafes.csv')
features = ['latitude', 'longitude', 'rating', 'review_count', 'price_min']
df[features] = df[features].fillna(df[features].mean())
X = df[features].to_numpy(dtype=np.float32)
n_data, dim = X.shape
query = X[:1].copy()
k = 10
faiss.omp_set_num_threads(4)
hasil_perbandingan = []

In [2]:
def bandingkan_model(metric='euclidean'):
    # Data dan query sama untuk ketiga model.
    data = X.copy()
    q = query.copy()
    if metric == 'cosine':
        faiss.normalize_L2(data)
        faiss.normalize_L2(q)

    hasil = []

    # 2. Annoy
    print('=== Annoy ===')
    ann_index = AnnoyIndex(dim, 'euclidean' if metric == 'euclidean' else 'angular')
    ann_index.set_seed(42)
    start = time.perf_counter()
    for i in range(n_data):
        ann_index.add_item(i, data[i])
    ann_index.build(10, n_jobs=4)
    build_time = time.perf_counter() - start

    start = time.perf_counter()
    neighbors = ann_index.get_nns_by_vector(q[0], k, include_distances=True)
    query_time = time.perf_counter() - start
    print('Build time:', build_time, 'detik')
    print('Query time:', query_time, 'detik')
    print('Neighbors:', neighbors[0][:5], '...')
    hasil.append({'Distance Metrics': metric.capitalize(), 'Model': 'Annoy',
                  'Jumlah data': n_data, 'Dimensi': dim, 'Index terdekat': neighbors[0],
                  'Build time (detik)': build_time, 'Query time (detik)': query_time})
    del ann_index

    # 3. FAISS (Flat Index)
    print('\n=== FAISS (Flat Index) ===')
    faiss_index = faiss.IndexFlatL2(dim) if metric == 'euclidean' else faiss.IndexFlatIP(dim)
    start = time.perf_counter()
    faiss_index.add(data)
    build_time = time.perf_counter() - start

    start = time.perf_counter()
    distances, indices = faiss_index.search(q, k)
    query_time = time.perf_counter() - start
    print('Build time:', build_time, 'detik')
    print('Query time:', query_time, 'detik')
    print('Neighbors:', indices[0][:5], '...')
    hasil.append({'Distance Metrics': metric.capitalize(), 'Model': 'FAISS Flat',
                  'Jumlah data': n_data, 'Dimensi': dim, 'Index terdekat': indices[0].tolist(),
                  'Build time (detik)': build_time, 'Query time (detik)': query_time})
    del faiss_index

    # 4. HNSW (hnswlib)
    print('\n=== HNSW (hnswlib) ===')
    hnsw_index = hnswlib.Index(space='l2' if metric == 'euclidean' else 'cosine', dim=dim)
    start = time.perf_counter()
    hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16, random_seed=42)
    hnsw_index.set_num_threads(4)
    hnsw_index.add_items(data)
    build_time = time.perf_counter() - start
    hnsw_index.set_ef(50)

    start = time.perf_counter()
    labels, distances = hnsw_index.knn_query(q, k=k)
    query_time = time.perf_counter() - start
    print('Build time:', build_time, 'detik')
    print('Query time:', query_time, 'detik')
    print('Neighbors:', labels[0][:5], '...')
    hasil.append({'Distance Metrics': metric.capitalize(), 'Model': 'HNSW',
                  'Jumlah data': n_data, 'Dimensi': dim, 'Index terdekat': labels[0].tolist(),
                  'Build time (detik)': build_time, 'Query time (detik)': query_time})
    return hasil

In [3]:
hasil_perbandingan.extend(bandingkan_model('euclidean'))

=== Annoy ===
Build time: 0.0018837579991668463 detik
Query time: 2.4699000277905725e-05 detik
Neighbors: [0, 85, 225, 201, 115] ...

=== FAISS (Flat Index) ===
Build time: 3.051999919989612e-05 detik
Query time: 2.6731999241746962e-05 detik
Neighbors: [  0  85 225 201 115] ...

=== HNSW (hnswlib) ===
Build time: 0.003542484999343287 detik
Query time: 2.5229001039406285e-05 detik
Neighbors: [  0  85 225 201 115] ...


In [4]:
hasil_perbandingan.extend(bandingkan_model('cosine'))

=== Annoy ===
Build time: 0.0023926410012791166 detik
Query time: 2.6110999897355214e-05 detik
Neighbors: [0, 85, 225, 115, 201] ...

=== FAISS (Flat Index) ===
Build time: 2.0650999431381933e-05 detik
Query time: 2.211299943155609e-05 detik
Neighbors: [  0  85 225 115 201] ...

=== HNSW (hnswlib) ===
Build time: 0.003076425000472227 detik
Query time: 2.610099909361452e-05 detik
Neighbors: [  0  85 225 115 201] ...


In [5]:
tabel_perbandingan = pd.DataFrame(hasil_perbandingan)
with pd.option_context('display.max_colwidth', None, 'display.float_format', '{:.8f}'.format):
    display(tabel_perbandingan)

,Distance Metrics,Model,Jumlah data,Dimensi,Index terdekat,Build time (detik),Query time (detik)
0,Euclidean,Annoy,302,5,"[0, 85, 225, 201, 115, 24, 289, 238, 149, 94]",0.00188376,0.00002470
1,Euclidean,FAISS Flat,302,5,"[0, 85, 225, 201, 115, 24, 289, 238, 149, 94]",0.00003052,0.00002673
2,Euclidean,HNSW,302,5,"[0, 85, 225, 201, 115, 24, 289, 238, 149, 94]",0.00354248,0.00002523
3,Cosine,Annoy,302,5,"[0, 85, 225, 115, 201, 24, 289, 238, 149, 94]",0.00239264,0.00002611
4,Cosine,FAISS Flat,302,5,"[0, 85, 225, 115, 201, 24, 289, 238, 149, 94]",0.00002065,0.00002211
5,Cosine,HNSW,302,5,"[0, 85, 225, 115, 201, 24, 289, 238, 149, 94]",0.00307643,0.00002610
